# Typewell Transformer v3 — encoder + decoder, GRU'suz

v2'nin (GRU + cross-attention) yerine tamamen attention tabanlı **seq2seq** mimari: typewell'i bir *encoder* okur, horizontal well bir *decoder* olarak işlenir ve her decoder bloğu typewell belleğine cross-attention yapar. İki yapısal kazanç:

1. **Bidirectional bağlam.** Test kuyusunun GR ve trajectory'si sonuna kadar verili; gizli olan yalnızca PS sonrası TVT. GRU soldan sağa causal'dı — t satırı gelecekteki GR pattern'ını göremiyordu. Decoder'da causal mask YOK: her satır kuyunun tamamını görür. TVT'nin döndüğü/kıvrıldığı kuyularda en büyük avantaj bu.
2. **Stateful makine kalkar.** Hidden state olmadığı için warmup segmentleri, hidden taşıma, segment başına optimizer step ve sol-pad'in hidden'ı kirletmesi sorunu tamamen yok. Episode = tek forward; padding `key_padding_mask` ile doğru maskelenir, batch kompozisyonu sonucu etkileyemez.

## Mimari

```
ENCODER (typewell)                       DECODER (horizontal well)
ince grid (2048, [GR, dtvt])             satirlar (T, 24)
  strided Conv1d x2 (4x sikistirma)        Linear -> row_h (T, d)
  -> 512 token + sin PE                    strided Conv1d (P=8) + sin PE -> (T/8, d)
  TransformerEncoder (2 katman)            ┌─ 4x TransformerDecoderLayer ──────┐
        |                                  │  self-attn (bidirectional)        │
     memory (512, d) ──────────────────────┤  cross-attn -> memory             │
        |                                  │  FFN                              │
        |                                  └───────────────────────────────────┘
        |                                  upsample + row_h + local Conv -> h (T, d)
        |                                        |
        +──────────────── pointer attn (tek kafa, locality + NCC bias)
        +──────────────── context cross-attn (satir seviyesi, 4 kafa)
                                                 |
                          prediction = anchor + soft_dtvt + residual_head(h, ctx, soft_dtvt)
```

- **Adresleme ve icerik cozunurlugu ayrildi.** Typewell 2048 noktalik ince gride resample edilir (512'de GR varyansinin ~%15'i kayboluyordu, 2048'de ~%3). Attention bellegi yine 512 token: ince sinyal decimation'la atilmaz, iki kademeli strided Conv (öğrenilmiş filtreler, örtüşen pencereler) ile 512 tokena sıkıştırılır. Pointer adresleri (dtvt) token merkezlerinde kalır — attention maliyeti degismez, icerik tam cozunurluk olur. NCC pattern'lari da ince gridden orneklenir.
- Horizontal self/cross attention **patch seviyesinde** (P=8): en uzun kuyu 7835 satır → ≤980 token, tam attention ucuz. Satır seviyesindeki ince detay `row_h` + küçük bir local Conv ile korunur; pointer ve context okuma satır seviyesinde kalır.
- v2'de doğrulanan her şey taşındı: ayrı dropout'suz pointer attention, öğrenilebilir locality bias (−dtvt²/2σ²), NCC korelasyon kanalı, pointer-residual çıktı, pointer + smoothness loss'ları, episode-bazlı GR kalibrasyonu.
- Satır seviyesindeki cross-attention'da dropout=0 (SDPA hızlı yolu her yerde çalışır); regularizasyon decoder katmanları ve head'de.
- Transformer'lar LR'a duyarlı olduğu için linear warmup + cosine decay schedule eklendi.

Veri pipeline'ı v1/v2 ile birebir aynı. Checkpoint: `typewell_transformer_v3_best.pt`.

In [ ]:
import copy
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from training import (
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    StatefulTVTWindowDataset,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
VALID_SIZE = 0.15
WINDOW_SIZE = 256      # dataset sinifi istiyor; v3'te segment KULLANILMAZ
CONTEXT_LENGTH = 256
MIN_FORECAST_LENGTH = 256
MIN_FORECAST_FRACTION = 0.40
REAL_PS_ANCHOR_PROB = 0.70
REAL_PS_FRACTION_RANGE = (0.20, 0.35)
ANCHORS_PER_WELL = 8
NUM_EPOCHS = 10

# v3: episode = tum kuyu tek forward. Bellek olceklenmesi satir sayisiyla
# oldugu icin batch v2'den kucuk tutulur.
EPISODE_BATCH_SIZE = 8

# Model boyutlari
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2   # typewell encoder
NUM_DECODER_LAYERS = 4   # horizontal decoder
FFN_DIM = 256
PATCH_SIZE = 8           # horizontal satirlar patch token'lara indirgenir
DROPOUT = 0.10

# Loss agirliklari (v2 ile ayni)
POINTER_LOSS_WEIGHT = 0.3
SMOOTHNESS_LOSS_WEIGHT = 0.2

# Optimizasyon: transformer LR schedule ister
BASE_LR = 3e-4
WARMUP_FRACTION = 0.05

CHECKPOINT_PATH = "typewell_transformer_v3_best.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# Veri pipeline'i — v1/v2 ile birebir ayni.

all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)
print(len(train_wells), len(valid_wells), len(test_wells))

In [ ]:
# Typewell bellegi + scaler'lar.
#
# v3 FARKI: typewell 2048 noktalik INCE gride resample edilir (512'de GR
# varyansinin ~%15'i kayboluyordu). Attention bellegi yine 512 token; ince
# grid modeldeki ogrenilebilir strided-Conv blogu ile sikistirilir. Burada
# uc sey uretilir:
#   - tw_gr_fine   (2048,) : ince grid GR — encoder'in ham girdisi
#   - tw_patterns  (512,13): NCC icin token merkezlerinde GR pencereleri.
#     Ince gridde ±12 ornek (25) alinip 2'de 1 seyreltilir -> 13 kanal;
#     ornek araligi ~0.9 ft, horizontal'in ~1 ft satir araligiyla uyumlu.
#   - tw_tvt_centers (512,): token merkezlerinin TVT'si (pointer adresleri).
#     Stride-4 toplam sikistirmada cikti token i, ince grid 4i hizasindadir.

TYPEWELL_FINE_POINTS = 2048
TW_DOWNSAMPLE = 4
TYPEWELL_TOKENS = TYPEWELL_FINE_POINTS // TW_DOWNSAMPLE  # 512
PATTERN_HALF_WINDOW = 12   # ince gridde ±12 ornek
NCC_WINDOW = PATTERN_HALF_WINDOW + 1  # 13 (25 ornek, stride 2)


def build_typewell_grid(typewell_df, num_points=TYPEWELL_FINE_POINTS):
    typewell = (
        typewell_df.dropna(subset=["TVT", "GR"])
        .sort_values("TVT")
        .drop_duplicates("TVT")
    )
    if len(typewell) < 2:
        raise ValueError("Typewell icinde yeterli TVT-GR verisi yok.")
    tvt_grid = np.linspace(
        typewell["TVT"].min(), typewell["TVT"].max(), num_points
    ).astype(np.float32)
    gr_grid = np.interp(
        tvt_grid,
        typewell["TVT"].to_numpy(dtype=float),
        typewell["GR"].to_numpy(dtype=float),
    ).astype(np.float32)
    return tvt_grid, gr_grid


def fit_typewell_gr_calibration(known_tvt, known_gr, tvt_grid, gr_grid):
    in_range = (known_tvt >= tvt_grid[0]) & (known_tvt <= tvt_grid[-1])
    known_tvt = known_tvt[in_range]
    known_gr = known_gr[in_range]
    if len(known_tvt) < 3:
        return 1.0, 0.0
    expected_gr = np.interp(known_tvt, tvt_grid, gr_grid)
    scale, offset = np.polyfit(expected_gr, known_gr, 1)
    if not np.isfinite(scale) or abs(scale) < 1e-8:
        return 1.0, 0.0
    return float(scale), float(offset)


def known_calibration_pairs(well_data, tvt_column, gr_column="GR"):
    known = well_data.loc[
        well_data[tvt_column].notna() & (well_data["GR_isna"] == 0),
        [tvt_column, gr_column],
    ]
    return (
        known[tvt_column].to_numpy(dtype=float),
        known[gr_column].to_numpy(dtype=float),
    )


def build_typewell_patterns(gr_grid):
    """Token merkezlerinde NCC pattern'lari: (TYPEWELL_TOKENS, NCC_WINDOW).

    Ince gridde her nokta icin ±PATTERN_HALF_WINDOW'luk pencere alinir,
    TW_DOWNSAMPLE'de bir satir (token merkezleri) ve pencere icinde 2'de
    bir sutun secilir; 25 ornek -> 13 kanal, merkez korunur.
    """
    padded = np.pad(gr_grid, PATTERN_HALF_WINDOW, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(
        padded, 2 * PATTERN_HALF_WINDOW + 1
    )
    return np.ascontiguousarray(
        windows[::TW_DOWNSAMPLE, ::2]
    ).astype(np.float32)


def attach_typewell_memory(prepared_wells):
    for well in tqdm(prepared_wells.values(), desc="Typewell memory", unit="well"):
        tvt_grid, gr_grid = build_typewell_grid(well.typewell_df)
        known_tvt, known_gr = known_calibration_pairs(
            well.data, "observed_TVT"
        )
        scale, offset = fit_typewell_gr_calibration(
            known_tvt, known_gr, tvt_grid, gr_grid
        )
        well.tw_tvt_fine = tvt_grid                             # (2048,)
        well.tw_gr_fine_raw = gr_grid                           # (2048,)
        well.tw_tvt_centers = np.ascontiguousarray(
            tvt_grid[::TW_DOWNSAMPLE]
        )                                                       # (512,)
        well.tw_patterns_raw = build_typewell_patterns(gr_grid) # (512, 13)
        well.tw_gr_fine_cal = gr_grid * scale + offset
        well.tw_patterns_cal = well.tw_patterns_raw * scale + offset
        well.tw_gr_calibration = (scale, offset)


attach_typewell_memory(train_wells)
attach_typewell_memory(valid_wells)
attach_typewell_memory(test_wells)

MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
for well in train_wells.values():
    feature_scaler.partial_fit(well.data[CONTINUOUS_FEATURE_COLUMNS])
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])
_gr_channel = CONTINUOUS_FEATURE_COLUMNS.index("GR")
gr_mean = float(feature_scaler.mean_[_gr_channel])
gr_scale = float(feature_scaler.scale_[_gr_channel])

GR_FEATURE_INDEX = RNN_FEATURE_COLUMNS.index("GR")


def apply_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, "GR_raw"] = well.data["GR"]
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(well.data[CONTINUOUS_FEATURE_COLUMNS])
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()
        well.data.loc[:, "observed_TVT_scaled"] = 0.0
        observed_mask = well.data["observed_TVT"].notna()
        well.data.loc[observed_mask, "observed_TVT_scaled"] = (
            well.data.loc[observed_mask, "observed_TVT"] - target_mean
        ) / target_scale
        well.tw_gr_fine_scaled = (
            (well.tw_gr_fine_cal - gr_mean) / gr_scale
        ).astype(np.float32)
        well.tw_patterns_scaled = (
            (well.tw_patterns_cal - gr_mean) / gr_scale
        ).astype(np.float32)


apply_scalers(train_wells)
apply_scalers(valid_wells)
apply_scalers(test_wells)
print(f"target: mean={target_mean:.1f}, scale={target_scale:.1f}")

In [ ]:
# Dataset — v2'den iki fark:
# 1) Segment iterasyonu yerine ds.features / ds.target / ds.loss_mask tum
#    kuyu tensorlari olarak dogrudan alinir (StatefulTVTWindowDataset zaten
#    tum kuyuyu kurup segmentleri sonradan dilimliyordu; dilimleri atliyoruz).
# 2) Typewell artik uc tensor: ince grid icerigi (tw_fine), NCC pattern'lari
#    (tw_patterns) ve token merkezi adresleri (tw_dtvt).


class TypewellStatefulTVTWindowDataset(StatefulTVTWindowDataset):
    """v3 typewell tensorlari:

    - tw_fine     (2048, 2): [GR_scaled, dtvt] ince gridde — encoder'in
      strided-Conv blogunun girdisi (icerik tam cozunurluk).
    - tw_patterns (512, 13): token merkezlerindeki NCC pattern'lari.
    - tw_dtvt     (512,)   : token merkezlerinin anchor'a gore dtvt'si
      (pointer adresleri / locality bias).
    """

    def __init__(
        self,
        *,
        tw_tvt_fine,
        tw_tvt_centers,
        tw_gr_fine_scaled,
        tw_patterns_scaled,
        target_scale,
        target_mean,
        **kwargs,
    ):
        super().__init__(**kwargs)
        anchor_row = self.prediction_start - 1
        anchor_tvt = float(self.data.loc[anchor_row, "TVT"])
        dtvt_centers = (
            (tw_tvt_centers.astype(np.float64) - anchor_tvt) / target_scale
        ).astype(np.float32)
        dtvt_fine = (
            (tw_tvt_fine.astype(np.float64) - anchor_tvt) / target_scale
        ).astype(np.float32)
        self.anchor_tvt = anchor_tvt
        self.anchor_scaled = torch.tensor(
            (anchor_tvt - target_mean) / target_scale, dtype=torch.float32
        )
        self.tw_dtvt = torch.from_numpy(dtvt_centers)
        self.tw_fine = torch.from_numpy(
            np.stack([tw_gr_fine_scaled, dtvt_fine], axis=1).astype(np.float32)
        )
        self.tw_patterns = torch.from_numpy(
            tw_patterns_scaled.astype(np.float32)
        )

    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        item["tw_fine"] = self.tw_fine
        item["tw_patterns"] = self.tw_patterns
        item["tw_dtvt"] = self.tw_dtvt
        item["anchor_scaled"] = self.anchor_scaled
        return item


def make_real_datasets(prepared_wells):
    return {
        well_id: TypewellStatefulTVTWindowDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            window_size=WINDOW_SIZE,
            context_length=CONTEXT_LENGTH,
            mode="real",
            prediction_mask_column="is_prediction_row",
            observed_context_column="observed_TVT_scaled",
            tw_tvt_fine=well.tw_tvt_fine,
            tw_tvt_centers=well.tw_tvt_centers,
            tw_gr_fine_scaled=well.tw_gr_fine_scaled,
            tw_patterns_scaled=well.tw_patterns_scaled,
            target_scale=target_scale,
            target_mean=target_mean,
        )
        for well_id, well in prepared_wells.items()
    }


def make_episode_dataset(well, prediction_start):
    """Sentetik anchor'li episode; kalibrasyon yalnizca anchor oncesiyle."""
    known = well.data.iloc[:prediction_start]
    known_tvt, known_gr = known_calibration_pairs(
        known, "TVT", gr_column="GR_raw"
    )
    scale, offset = fit_typewell_gr_calibration(
        known_tvt, known_gr, well.tw_tvt_fine, well.tw_gr_fine_raw
    )
    gr_fine_scaled = (
        (well.tw_gr_fine_raw * scale + offset - gr_mean) / gr_scale
    ).astype(np.float32)
    patterns_scaled = (
        (well.tw_patterns_raw * scale + offset - gr_mean) / gr_scale
    ).astype(np.float32)

    return TypewellStatefulTVTWindowDataset(
        data=well.data,
        feature_columns=RNN_FEATURE_COLUMNS,
        target_column="TVT_scaled",
        window_size=WINDOW_SIZE,
        context_length=CONTEXT_LENGTH,
        mode="synthetic",
        prediction_start=prediction_start,
        tw_tvt_fine=well.tw_tvt_fine,
        tw_tvt_centers=well.tw_tvt_centers,
        tw_gr_fine_scaled=gr_fine_scaled,
        tw_patterns_scaled=patterns_scaled,
        target_scale=target_scale,
        target_mean=target_mean,
    )


valid_datasets = make_real_datasets(valid_wells)
test_datasets = make_real_datasets(test_wells)

_ds = next(iter(test_datasets.values()))
print(f"tum kuyu features: {tuple(_ds.features.shape)}, "
      f"loss satiri: {int(_ds.loss_mask.sum())}")
print(f"tw_fine: {tuple(_ds.tw_fine.shape)}, "
      f"tw_patterns: {tuple(_ds.tw_patterns.shape)}, "
      f"tw_dtvt: {tuple(_ds.tw_dtvt.shape)}")

In [ ]:
def sinusoidal_positions(length, dim, device):
    position = torch.arange(length, device=device, dtype=torch.float32)
    div = torch.exp(
        torch.arange(0, dim, 2, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / dim)
    )
    pe = torch.zeros(length, dim, device=device)
    pe[:, 0::2] = torch.sin(position[:, None] * div)
    pe[:, 1::2] = torch.cos(position[:, None] * div)
    return pe


class TypewellTransformerModel(nn.Module):
    """Encoder-decoder: typewell'i encoder okur, horizontal well decoder'dir.

    - Encoder: ince grid (2048, [GR, dtvt]) -> iki kademeli strided Conv1d
      (k=9, s=2; toplam 4x sikistirma, ogrenilebilir) -> 512 token + sin PE
      -> TransformerEncoder. Cikti `memory` (512, d). Icerik tam
      cozunurluktan gelir; pointer adresleri token merkezlerinde kalir
      (cikti token i ~ ince grid 4i, cunku k=9/s=2/p=4 conv'un cikti merkezi
      girdinin 2i'sine denk gelir).
    - Decoder: satirlar Linear ile embed edilir (row_h), strided Conv1d ile
      P satirlik patch token'lara indirgenir (T/P adet). Her
      TransformerDecoderLayer once patch'ler arasi BIDIRECTIONAL self-attn
      (causal mask yok — gelecekteki GR gorulur), sonra typewell memory'ye
      cross-attn, sonra FFN uygular.
    - Patch cikti satirlara upsample edilip row_h ile birlestirilir; kucuk
      bir local Conv satir seviyesi detayi geri kazandirir.
    - Pointer (v2'den): ayri tek kafali, dropout'suz attention; logit =
      dot/sqrt(d) + locality(-dtvt^2/2sigma^2, sigma ogrenilebilir) +
      corr_gain * NCC(horizontal GR penceresi, ince gridden orneklenen
      token pattern'i).
    - Cikti: prediction = anchor_scaled + soft_dtvt + residual.
    """

    def __init__(
        self,
        input_size: int,
        tw_fine_channels: int = 2,
        d_model: int = D_MODEL,
        num_heads: int = NUM_HEADS,
        num_encoder_layers: int = NUM_ENCODER_LAYERS,
        num_decoder_layers: int = NUM_DECODER_LAYERS,
        ffn_dim: int = FFN_DIM,
        patch_size: int = PATCH_SIZE,
        dropout: float = DROPOUT,
        gr_feature_index: int = GR_FEATURE_INDEX,
        corr_window: int = NCC_WINDOW,
        locality_sigma_init: float = 0.25,
    ):
        super().__init__()
        self.d_model = d_model
        self.patch_size = patch_size
        self.gr_feature_index = gr_feature_index
        self.corr_window = corr_window

        # --- encoder (typewell) ---
        # Ince grid -> 512 token: ogrenilebilir sikistirma (decimation degil).
        self.tw_downsample = nn.Sequential(
            nn.Conv1d(tw_fine_channels, d_model, kernel_size=9,
                      stride=2, padding=4),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=9,
                      stride=2, padding=4),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.tw_encoder = nn.TransformerEncoder(
            encoder_layer, num_encoder_layers
        )

        # --- decoder (horizontal) ---
        self.row_embed = nn.Linear(input_size, d_model)
        self.patch_conv = nn.Conv1d(
            d_model, d_model, kernel_size=patch_size, stride=patch_size
        )
        decoder_layer = nn.TransformerDecoderLayer(
            d_model, num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer, num_decoder_layers
        )

        self.fuse_norm = nn.LayerNorm(d_model)
        self.local_conv = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=9, padding=4,
                      groups=d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=1),
        )

        # --- satir seviyesi typewell okumalari ---
        self.context_attn = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True, dropout=0.0
        )
        self.pointer_q = nn.Linear(d_model, d_model)
        self.pointer_k = nn.Linear(d_model, d_model)
        self.log_sigma = nn.Parameter(
            torch.tensor(math.log(locality_sigma_init))
        )
        self.corr_gain = nn.Parameter(torch.tensor(1.0))

        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 2 + 1),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2 + 1, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def _gr_correlation(self, x, tw_patterns):
        w = self.corr_window
        gr = x[..., self.gr_feature_index]
        padded = F.pad(gr.unsqueeze(1), (w - 1, 0), mode="replicate")
        windows = padded.squeeze(1).unfold(1, w, 1)
        windows = windows - windows.mean(-1, keepdim=True)
        windows = windows / windows.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        patterns = tw_patterns - tw_patterns.mean(-1, keepdim=True)
        patterns = patterns / patterns.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return torch.einsum("btw,bmw->btm", windows, patterns)

    def forward(self, x, tw_fine, tw_patterns, tw_dtvt, anchor_scaled,
                pad_mask=None):
        # x: (B, T, F) — T, patch_size'in kati olacak sekilde pad'lenmis.
        # tw_fine: (B, 2048, 2), tw_patterns: (B, 512, 13),
        # tw_dtvt: (B, 512). pad_mask: (B, T) bool, True = padding satiri.
        B, T, _ = x.shape
        P = self.patch_size
        num_patches = T // P

        # Encoder: ince grid -> 512 token -> typewell memory
        memory = self.tw_downsample(
            tw_fine.transpose(1, 2)
        ).transpose(1, 2)                                     # (B, M, d)
        memory = memory + sinusoidal_positions(
            memory.shape[1], self.d_model, memory.device
        )
        memory = self.tw_encoder(memory)                      # (B, M, d)

        # Decoder girdisi: satir embed -> patch token
        row_h = self.row_embed(x)                             # (B, T, d)
        patches = self.patch_conv(
            row_h.transpose(1, 2)
        ).transpose(1, 2)                                     # (B, T/P, d)
        patches = patches + sinusoidal_positions(
            num_patches, self.d_model, patches.device
        )

        patch_pad = None
        if pad_mask is not None:
            patch_pad = pad_mask.view(B, num_patches, P).all(-1)

        decoded = self.decoder(
            patches, memory, tgt_key_padding_mask=patch_pad
        )                                                     # (B, T/P, d)

        # Satirlara geri: upsample + row detayi + local mixing
        upsampled = decoded.repeat_interleave(P, dim=1)       # (B, T, d)
        h = self.fuse_norm(row_h + upsampled)
        h = h + self.local_conv(h.transpose(1, 2)).transpose(1, 2)

        # Context okuma (satir seviyesi)
        context, _ = self.context_attn(
            h, memory, memory, need_weights=False
        )                                                     # (B, T, d)

        # Pointer (satir seviyesi)
        logits = torch.einsum(
            "bth,bmh->btm", self.pointer_q(h), self.pointer_k(memory)
        ) / math.sqrt(self.d_model)
        sigma = self.log_sigma.exp()
        locality = -(tw_dtvt ** 2).unsqueeze(1) / (2.0 * sigma ** 2)
        corr = self._gr_correlation(x, tw_patterns)
        logits = logits + locality + self.corr_gain * corr

        pointer_alpha = torch.softmax(logits, dim=-1)         # (B, T, M)
        soft_dtvt = torch.einsum("btm,bm->bt", pointer_alpha, tw_dtvt)

        head_input = torch.cat(
            [h, context, soft_dtvt.unsqueeze(-1)], dim=-1
        )
        residual = self.head(head_input).squeeze(-1)
        predictions = anchor_scaled.unsqueeze(-1) + soft_dtvt + residual
        return predictions, pointer_alpha, soft_dtvt

In [ ]:
model = TypewellTransformerModel(
    input_size=len(RNN_FEATURE_COLUMNS) + 2,
).to(device)

# --- smoke test: tek kuyu, tum satirlar tek forward ---
_ds = next(iter(test_datasets.values()))
_T = len(_ds.features)
_T_pad = math.ceil(_T / PATCH_SIZE) * PATCH_SIZE

_x = torch.zeros(1, _T_pad, _ds.features.shape[1])
_x[0, :_T] = _ds.features
_pad = torch.ones(1, _T_pad, dtype=torch.bool)
_pad[0, :_T] = False

model.eval()
with torch.no_grad():
    _pred, _alpha, _soft = model(
        _x.to(device),
        _ds.tw_fine.unsqueeze(0).to(device),
        _ds.tw_patterns.unsqueeze(0).to(device),
        _ds.tw_dtvt.unsqueeze(0).to(device),
        _ds.anchor_scaled.unsqueeze(0).to(device),
        pad_mask=_pad.to(device),
    )

assert _pred.shape == (1, _T_pad)
assert _alpha.shape == (1, _T_pad, TYPEWELL_TOKENS)
assert torch.isfinite(_pred).all() and torch.isfinite(_alpha).all()
assert torch.allclose(
    _alpha.sum(-1), torch.ones_like(_alpha.sum(-1)), atol=1e-5
)

# Train modunda da (dropout aktifken) pointer normalize kalmali
model.train()
_pred_tr, _alpha_tr, _ = model(
    _x.to(device),
    _ds.tw_fine.unsqueeze(0).to(device),
    _ds.tw_patterns.unsqueeze(0).to(device),
    _ds.tw_dtvt.unsqueeze(0).to(device),
    _ds.anchor_scaled.unsqueeze(0).to(device),
    pad_mask=_pad.to(device),
)
assert torch.allclose(
    _alpha_tr.sum(-1), torch.ones_like(_alpha_tr.sum(-1)), atol=1e-5
)

n_params = sum(p.numel() for p in model.parameters())
print(f"kuyu uzunlugu: {_T} satir -> {_T_pad // PATCH_SIZE} patch token")
print(f"pred: {tuple(_pred.shape)}, alpha: {tuple(_alpha.shape)}")
print(f"parameters: {n_params:,} (v2: 125,253)")

In [ ]:
# Egitim/degerlendirme. v2'den farklar:
# - Segment dongusu ve hidden tasima YOK: episode = tek forward.
# - Episode'lar sagdan pad'lenir; pad satirlari pad_mask ile attention'dan
#   ve loss'tan dislanir (sol-pad/hidden sorunu yapisal olarak yok).
# - Optimizer step batch basina; LR linear warmup + cosine decay.


def masked_mse_loss(predictions, targets, loss_mask):
    return ((predictions - targets) ** 2)[loss_mask].mean()


def sample_prediction_start(well, rng):
    num_rows = len(well.data)
    min_forecast = max(
        MIN_FORECAST_LENGTH,
        int(num_rows * MIN_FORECAST_FRACTION),
    )
    first = CONTEXT_LENGTH
    last = num_rows - min_forecast
    if last < first:
        return None
    if rng.random() < REAL_PS_ANCHOR_PROB:
        low = max(first, int(num_rows * REAL_PS_FRACTION_RANGE[0]))
        high = min(last, int(num_rows * REAL_PS_FRACTION_RANGE[1]))
        if low <= high:
            return rng.randint(low, high)
    return rng.randint(first, last)


def build_episode_batch(episode_datasets):
    """Episode'lari sagdan pad'leyip tek batch'e paketler (PS hizalama
    gerekmez; attention pozisyondan bagimsiz, pad'ler maskelenir)."""
    lengths = [len(ds.features) for ds in episode_datasets]
    total_len = math.ceil(max(lengths) / PATCH_SIZE) * PATCH_SIZE

    batch_size = len(episode_datasets)
    feature_dim = episode_datasets[0].features.shape[1]
    features = torch.zeros(batch_size, total_len, feature_dim)
    targets = torch.zeros(batch_size, total_len)
    loss_mask = torch.zeros(batch_size, total_len, dtype=torch.bool)
    pad_mask = torch.ones(batch_size, total_len, dtype=torch.bool)

    for i, ds in enumerate(episode_datasets):
        rows = len(ds.features)
        features[i, :rows] = ds.features
        targets[i, :rows] = ds.target
        loss_mask[i, :rows] = ds.loss_mask
        pad_mask[i, :rows] = False

    return {
        "features": features,
        "targets": targets,
        "loss_mask": loss_mask,
        "pad_mask": pad_mask,
        "tw_fine": torch.stack([ds.tw_fine for ds in episode_datasets]),
        "tw_patterns": torch.stack(
            [ds.tw_patterns for ds in episode_datasets]
        ),
        "tw_dtvt": torch.stack([ds.tw_dtvt for ds in episode_datasets]),
        "anchor_scaled": torch.stack(
            [ds.anchor_scaled for ds in episode_datasets]
        ),
    }


def run_batch(model, batch, optimizer=None, scheduler=None):
    features = batch["features"].to(device)
    targets = batch["targets"].to(device)
    loss_mask = batch["loss_mask"].to(device)
    pad_mask = batch["pad_mask"].to(device)
    tw_fine = batch["tw_fine"].to(device)
    tw_patterns = batch["tw_patterns"].to(device)
    tw_dtvt = batch["tw_dtvt"].to(device)
    anchor_scaled = batch["anchor_scaled"].to(device)

    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
        predictions, _, soft_dtvt = model(
            features, tw_fine, tw_patterns, tw_dtvt, anchor_scaled,
            pad_mask=pad_mask,
        )
        task_loss = masked_mse_loss(predictions, targets, loss_mask)

        true_dtvt = targets - anchor_scaled.unsqueeze(-1)
        pointer_loss = masked_mse_loss(soft_dtvt, true_dtvt, loss_mask)

        pair_mask = loss_mask[:, 1:] & loss_mask[:, :-1]
        if pair_mask.any():
            jumps = (soft_dtvt[:, 1:] - soft_dtvt[:, :-1])[pair_mask]
            smooth_loss = (jumps ** 2).mean()
        else:
            smooth_loss = soft_dtvt.new_zeros(())

        loss = (
            task_loss
            + POINTER_LOSS_WEIGHT * pointer_loss
            + SMOOTHNESS_LOSS_WEIGHT * smooth_loss
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        predictions = predictions.detach()
    else:
        predictions, _, _ = model(
            features, tw_fine, tw_patterns, tw_dtvt, anchor_scaled,
            pad_mask=pad_mask,
        )

    errors_tvt = (predictions - targets) * target_scale
    squared_error_sum = float(torch.sum(errors_tvt[loss_mask] ** 2).item())
    observation_count = int(loss_mask.sum().item())
    return squared_error_sum, observation_count


def train_one_epoch(model, prepared_wells, optimizer, scheduler, rng, epoch=None):
    model.train()

    episodes = []
    for well_id, well in prepared_wells.items():
        for _ in range(ANCHORS_PER_WELL):
            prediction_start = sample_prediction_start(well, rng)
            if prediction_start is not None:
                episodes.append((well_id, prediction_start))
    episodes.sort(key=lambda item: len(prepared_wells[item[0]].data))

    bucket_size = EPISODE_BATCH_SIZE * 4
    for i in range(0, len(episodes), bucket_size):
        bucket = episodes[i : i + bucket_size]
        rng.shuffle(bucket)
        episodes[i : i + bucket_size] = bucket

    batches = [
        episodes[i : i + EPISODE_BATCH_SIZE]
        for i in range(0, len(episodes), EPISODE_BATCH_SIZE)
    ]
    rng.shuffle(batches)

    squared_error_sum = 0.0
    observation_count = 0
    progress = tqdm(
        batches,
        desc=f"Epoch {epoch}" if epoch is not None else "Train",
        unit="batch",
        leave=False,
    )
    for batch_episodes in progress:
        episode_datasets = [
            make_episode_dataset(prepared_wells[well_id], prediction_start)
            for well_id, prediction_start in batch_episodes
        ]
        batch = build_episode_batch(episode_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer, scheduler)
        squared_error_sum += sq_err
        observation_count += n_obs

        if observation_count > 0:
            progress.set_postfix(
                running_rmse=f"{np.sqrt(squared_error_sum / observation_count):.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

    if observation_count == 0:
        raise ValueError("Training epoch'unda loss satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))


@torch.no_grad()
def evaluate_real_ps(model, datasets, batch_size=EPISODE_BATCH_SIZE):
    model.eval()

    ordered = sorted(datasets.values(), key=lambda ds: len(ds.features))
    batches = [
        ordered[i : i + batch_size]
        for i in range(0, len(ordered), batch_size)
    ]

    squared_error_sum = 0.0
    observation_count = 0
    for batch_datasets in tqdm(
        batches, desc="Eval", unit="batch", leave=False
    ):
        batch = build_episode_batch(batch_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer=None)
        squared_error_sum += sq_err
        observation_count += n_obs

    if observation_count == 0:
        raise ValueError("Gercek PS sonrasinda skorlanacak satir yok.")
    return float(np.sqrt(squared_error_sum / observation_count))

In [ ]:
# Optimizer + LR schedule (linear warmup -> cosine decay) ve egitim dongusu.

optimizer = torch.optim.AdamW(
    model.parameters(), lr=BASE_LR, weight_decay=1e-4
)

steps_per_epoch = math.ceil(
    len(train_wells) * ANCHORS_PER_WELL / EPISODE_BATCH_SIZE
)
total_steps = NUM_EPOCHS * steps_per_epoch
warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))


def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"steps/epoch ~{steps_per_epoch}, warmup {warmup_steps} step")

history = []
best_valid_rmse = float("inf")
best_state = None
anchor_rng = random.Random(SEED)

for epoch in range(1, NUM_EPOCHS + 1):
    train_rmse = train_one_epoch(
        model, train_wells, optimizer, scheduler, anchor_rng, epoch=epoch
    )
    valid_rmse = evaluate_real_ps(model, valid_datasets)
    history.append({
        "epoch": epoch,
        "synthetic_train_rmse": train_rmse,
        "real_ps_valid_rmse": valid_rmse,
    })
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch:02d} | synthetic train RMSE: {train_rmse:.4f} | "
        f"real-PS valid RMSE: {valid_rmse:.4f} | "
        f"sigma={model.log_sigma.exp().item():.3f} | "
        f"corr_gain={model.corr_gain.item():.3f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), CHECKPOINT_PATH)
test_rmse = evaluate_real_ps(model, test_datasets, batch_size=1)
print(f"Typewell transformer v3 reserved test RMSE: {test_rmse:.4f}")
pd.DataFrame(history)

In [ ]:
import matplotlib.pyplot as plt

# Pointer attention haritasi + tahmin. Episode tek forward oldugu icin
# toplama dongusu de yok.


@torch.no_grad()
def collect_well_outputs(model, dataset):
    model.eval()
    T = len(dataset.features)
    T_pad = math.ceil(T / PATCH_SIZE) * PATCH_SIZE

    x = torch.zeros(1, T_pad, dataset.features.shape[1])
    x[0, :T] = dataset.features
    pad = torch.ones(1, T_pad, dtype=torch.bool)
    pad[0, :T] = False

    predictions, alpha, soft_dtvt = model(
        x.to(device),
        dataset.tw_fine.unsqueeze(0).to(device),
        dataset.tw_patterns.unsqueeze(0).to(device),
        dataset.tw_dtvt.unsqueeze(0).to(device),
        dataset.anchor_scaled.unsqueeze(0).to(device),
        pad_mask=pad.to(device),
    )
    return (
        alpha.squeeze(0)[:T].cpu(),
        predictions.squeeze(0)[:T].cpu(),
        soft_dtvt.squeeze(0)[:T].cpu(),
    )


_well_id = max(
    test_datasets,
    key=lambda wid: int(test_wells[wid].data["is_prediction_row"].sum()),
)
_ds = test_datasets[_well_id]
_alpha_map, _preds, _softs = collect_well_outputs(model, _ds)

_well = test_wells[_well_id]
_true_tvt = _well.data["TVT"].to_numpy()
_pred_tvt = _preds.numpy() * target_scale + target_mean
_pointer_tvt = _softs.numpy() * target_scale + _ds.anchor_tvt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].imshow(
    _alpha_map.numpy().T,
    aspect="auto",
    origin="lower",
    extent=[
        0, len(_alpha_map),
        _well.tw_tvt_centers[0], _well.tw_tvt_centers[-1],
    ],
    cmap="viridis",
)
axes[0].plot(np.arange(len(_true_tvt)), _true_tvt, "r-", lw=0.8, label="true TVT")
axes[0].axvline(_ds.prediction_start, color="w", ls="--", lw=1, label="PS")
axes[0].set_ylabel("typewell TVT")
axes[0].set_title(f"Pointer attention map (v3 transformer) — {_well_id}")
axes[0].legend(loc="upper right")

axes[1].plot(_true_tvt, label="true TVT", lw=1)
axes[1].plot(_pred_tvt, label="predicted TVT", lw=1)
axes[1].plot(_pointer_tvt, label="pointer (anchor + soft_dtvt)", lw=0.8, alpha=0.7)
axes[1].axvline(_ds.prediction_start, color="k", ls="--", lw=1, label="PS")
axes[1].set_xlabel("row (MD sirasi)")
axes[1].set_ylabel("TVT")
axes[1].legend()

plt.tight_layout()
plt.show()

_pred_region = slice(_ds.prediction_start, len(_true_tvt))
_rmse = float(np.sqrt(np.mean(
    (_pred_tvt[_pred_region] - _true_tvt[_pred_region]) ** 2
)))
_pointer_rmse = float(np.sqrt(np.mean(
    (_pointer_tvt[_pred_region] - _true_tvt[_pred_region]) ** 2
)))
print(f"{_well_id} prediction-region RMSE: {_rmse:.3f} "
      f"(pointer tek basina: {_pointer_rmse:.3f})")